# ChaoSSM — Phase 22: why the deep-chaos trained-input configuration diverges

Addresses **Reviewer 1 comment 3** and **Reviewer 2 comment 3**: the deep-chaos trained-input arm
produces non-finite loss on 18/20 seeds; what causes it, is it gradient blow-up through the
Jacobians / shear at the fold kinks, and do standard mitigations (smaller step size, tighter
gradient clipping, a smoothed fold) help?

**The configuration.** The fair test's *trained-input* condition freezes the recurrence but trains
the input projection `B`, the encoder and the readout (the generalized-synchronization setting).

**Where the divergence is — and is not.** The triangle fold maps onto [0,1] by construction, so the
*forward* state stays bounded; the manuscript already reports verifying that neither the fold nor
its Jacobian produces non-finite values, and accordingly does **not** attribute the divergence to
unbounded state dynamics (an explanation an earlier draft speculated and then withdrew). This
notebook takes that as the starting point and tests the remaining candidate: the **backward** pass.
At the deep-chaos calibration (λ≈+0.30) the frozen driven recurrence is genuinely expanding — its
driven-state Jacobian dh_t/dh_{t-1} has top singular value >1 — so backpropagation through ~64 time
steps multiplies these expanding Jacobians and the gradient grows geometrically. The hypothesis:
**exploding BPTT gradients through the positive-λ recurrence, with the forward state bounded
throughout.** We separately ask whether the triangle-fold kinks (derivative jumping between ±1) are
a necessary ingredient or merely incidental.

**What this notebook does.**
1. **Instrumented reproduction** — trains the deep-chaos trained-input arm over many seeds, logging
   per-step loss and gradient norm, the driven-Jacobian top singular value, and the exact step of
   the first non-finite value. It checks the forward loss at initialization (expected finite — the
   bounded-state claim) and localizes the first non-finite value to the forward or the backward
   pass. Compares against the *edge* arm (which does not diverge).
2. **Mitigation sweep** — baseline vs smaller learning rate, tighter gradient clipping, a smoothed
   fold (continuous derivative, same 1-Lipschitz bound), and truncated BPTT (caps the number of
   multiplied Jacobians), each over several seeds, reporting the divergence rate and any recovered
   accuracy.

Note on detection: a non-finite *gradient* is what diverges first here; `clip_grad_norm_` silently
rescales a non-finite total norm to zero, so a loss-only check (as in the Phase 20 trainer) can miss
it and limp along on effectively-zero updates. We therefore flag the step at which **either** the
loss **or** the gradient norm first becomes non-finite.

Runtime: ~30–50 min on a T4. Resumable.

## Setup

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase22_divergence_forensics"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78); print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] Local disk, lost on recycle. Run the rescue cell or download."); print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase22_divergence_forensics


## Configuration

In [ ]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    steps:int=1500; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    gamma_init:float=1.0; r_cap_map:float=4.0
    lr:float=1e-3; grad_clip:float=1.0
    fc_length:int=64; fc_bins:int=32
    lambda_target:dict=field(default_factory=lambda: dict(edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    n_seeds_repro:int=12       # instrumented reproduction seeds
    n_seeds_mit:int=8          # mitigation-sweep seeds
    jac_every:int=25           # probe the driven-Jacobian top singular value every N steps
CFG=Config()
if SMOKE_TEST:
    CFG.steps=60; CFG.calib_steps=30; CFG.calib_warm=10; CFG.n_seeds_repro=3; CFG.n_seeds_mit=2
    print("*** SMOKE TEST — results meaningless ***")
print(f"repro seeds={CFG.n_seeds_repro}, mitigation seeds={CFG.n_seeds_mit}, steps={CFG.steps}")

repro seeds=12, mitigation seeds=8, steps=1500


## Cell with a configurable fold (hard or smoothed) — recurrence frozen, B trainable

In [ ]:
def triangle_fold_hard(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)
def triangle_fold_smooth(x,eps):
    """Smooth triangle fold: |.| replaced by sqrt(.^2+eps). eps->0 recovers the hard fold,
    but the derivative is continuous everywhere (no kink), removing the fold-kink shear."""
    return 1.0-torch.sqrt((torch.remainder(x,2.0)-1.0)**2+eps)

class ChaosCell(nn.Module):
    def __init__(self,cfg,fold_eps=0.0):
        super().__init__(); self.N=cfg.state_dim; self.r_cap=float(cfg.r_cap_map); self.fold_eps=fold_eps
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def fold(self,x): return triangle_fold_hard(x) if self.fold_eps<=0 else triangle_fold_smooth(x,self.fold_eps)
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return self.fold(self.get_gamma()*u)
    def step(self,h,x): return self.fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0):
    steps,warm,N=cfg.calib_steps,cfg.calib_warm,cfg.state_dim; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold_hard(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

class TrainedInputReservoir(nn.Module):
    """Recurrence frozen; encoder, B and readout TRAINED (the generalized-synchronization setting)."""
    def __init__(self,cfg,cell,out_dim,tbptt=0):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell; self.tbptt=int(tbptt)
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        cell.freeze_recurrence()                 # B stays trainable
    def forward(self,x):
        e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); H=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t])
            if self.tbptt>0 and (t+1)%self.tbptt==0: h=h.detach()
            H.append(h)
        return self.readout(self.ln(torch.stack(H,1)))

## Task + the instrumented driven-Jacobian probe

In [ ]:
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt

@torch.no_grad()
def driven_jacobian_top_sv(model, cfg, seed=0, n=30):
    """Top singular value of the driven-state Jacobian dh_t/dh_{t-1}, averaged over a short driven
    run with the CURRENT (training) encoder and B. >1 means the driven recurrence expands -> BPTT
    gradients grow geometrically. NaN-safe."""
    cell=model.cell; dev=cell.W_mix.device; N=cfg.state_dim
    g=torch.Generator(device=dev); g.manual_seed(seed)
    inp,_=make_forecast(cfg,1,dev,g); e=model.inp(inp.unsqueeze(-1))[0]
    h=cell.init_state(1,dev,torch.float32).squeeze(0); svs=[]
    def dstep(z,xv): return cell.fold(cell.phi(z)+xv@cell.B.T)
    for t in range(min(n,e.shape[0])):
        xt=e[t]
        try:
            J=torch.autograd.functional.jacobian(lambda z: dstep(z,xt), h)
            sv=float(torch.linalg.svdvals(J)[0])
            if math.isfinite(sv): svs.append(sv)
        except Exception: pass
        h=dstep(h,xt)
        if not torch.isfinite(h).all(): break
    return float(np.mean(svs)) if svs else float("nan")

## Instrumented training: log grad norm, detect the first non-finite step

In [ ]:
def make_lr_lambda(cfg,total,lr_scale=1.0):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def train_instrumented(cfg, regime, seed, lr=None, grad_clip=None, fold_eps=0.0, tbptt=0, probe=True):
    """tbptt>0 detaches the hidden state every `tbptt` steps (truncated BPTT), capping how many
    expanding driven-Jacobians multiply in a single gradient."""
    lr=cfg.lr if lr is None else lr; grad_clip=cfg.grad_clip if grad_clip is None else grad_clip
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    cell=ChaosCell(cfg,fold_eps=fold_eps).to(DEVICE); _,lam=calibrate(cell,cfg,regime,seed=seed)
    model=TrainedInputReservoir(cfg,cell,cfg.fc_bins,tbptt=tbptt).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    # (i) initialization forward pass: does the frozen expanding map overflow before any step?
    with torch.no_grad():
        inp0,tgt0=make_forecast(cfg,cfg.batch,DEVICE,gen)
        lg0=model(inp0); L0=F.cross_entropy(lg0.reshape(-1,lg0.shape[-1]),tgt0.reshape(-1))
    init_nonfinite=not math.isfinite(float(L0))
    first_nan=-1; gmax=0.0; gnorms=[]; jac_svs=[]
    for step in range(cfg.steps):
        inp,tgt=make_forecast(cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,grad_clip).item()
        gnorms.append(gn); gmax=max(gmax,gn if math.isfinite(gn) else gmax)
        if probe and step%cfg.jac_every==0:
            jac_svs.append(driven_jacobian_top_sv(model,cfg,seed=seed))
        if not (math.isfinite(L.item()) and math.isfinite(gn)):
            first_nan=step; break
        opt.step(); sched.step()
    diverged=first_nan>=0
    # held-out accuracy only if it survived
    acc=float("nan")
    if not diverged:
        ge=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); ge.manual_seed(cfg.fc_bins+777)
        ei,et=make_forecast(cfg,1500,DEVICE,ge)
        with torch.no_grad():
            pred=model(ei).argmax(-1); acc=float((pred==et).float().mean())
    phase=("init" if (diverged and init_nonfinite) else "train" if diverged else "none")
    return dict(regime=regime,seed=seed,lam=lam,diverged=diverged,init_nonfinite=init_nonfinite,
                divergence_phase=phase,first_nan_step=first_nan,grad_norm_max=gmax,acc=acc,
                jac_sv_mean=float(np.nanmean(jac_svs)) if jac_svs else float("nan"),
                jac_sv_last=float(jac_svs[-1]) if jac_svs else float("nan"),
                grad_norm_trace=json.dumps([round(x,3) for x in gnorms[:200]]))

## Part 1 — instrumented reproduction: deep (diverges) vs edge (does not)

In [ ]:
CKPT1=os.path.join(RESULTS,"phase22_reproduction.csv")
COLS1=["regime","seed","lam","diverged","init_nonfinite","divergence_phase","first_nan_step","grad_norm_max","acc","jac_sv_mean","jac_sv_last","grad_norm_trace"]
if os.path.exists(CKPT1):
    rows1=pd.read_csv(CKPT1).to_dict("records"); done1={(r["regime"],int(r["seed"])) for r in rows1}
    print(f"[resume] {len(rows1)} reproduction runs done")
else: rows1=[]; done1=set()
def _f1(): pd.DataFrame(rows1)[COLS1].to_csv(CKPT1,index=False)
plan1=[(reg,s) for reg in ("deep","edge") for s in range(CFG.n_seeds_repro)]
for reg,s in [k for k in plan1 if k not in done1]:
    r=train_instrumented(CFG,reg,s); rows1.append(r); _f1()
    print(f"[repro] {reg:4s} s={s} lam={r['lam']:+.3f} "
          +("DIVERGED(%s)@%d"%(r['divergence_phase'],r['first_nan_step']) if r['diverged'] else f"ok acc={r['acc']:.4f}")
          +f" | grad_max={r['grad_norm_max']:.1e} jac_sv={r['jac_sv_mean']:.3f}")
RES1=pd.DataFrame(rows1)[COLS1]
print("\ndivergence rate:")
for reg in ("deep","edge"):
    d=RES1[RES1.regime==reg]
    print(f"  {reg:4s}: {int(d.diverged.sum())}/{len(d)} diverged "
          f"(init-overflow {int((d.divergence_phase=='init').sum())}, train-blowup {int((d.divergence_phase=='train').sum())}), "
          f"driven-Jacobian top sv = {d.jac_sv_mean.mean():.3f}, grad_norm_max = {d.grad_norm_max.max():.1e}")

[repro] deep s=0 lam=+0.272 DIVERGED(train)@188 | grad_max=1.2e+19 jac_sv=2.962
[repro] deep s=1 lam=+0.293 DIVERGED(train)@0 | grad_max=0.0e+00 jac_sv=3.052
[repro] deep s=2 lam=+0.321 DIVERGED(train)@87 | grad_max=1.5e+19 jac_sv=3.043
[repro] deep s=3 lam=+0.306 DIVERGED(train)@149 | grad_max=1.1e+19 jac_sv=3.026
[repro] deep s=4 lam=+0.273 DIVERGED(train)@64 | grad_max=1.7e+19 jac_sv=2.877
[repro] deep s=5 lam=+0.295 DIVERGED(train)@601 | grad_max=1.2e+19 jac_sv=2.949
[repro] deep s=6 lam=+0.302 DIVERGED(train)@657 | grad_max=7.1e+18 jac_sv=3.022
[repro] deep s=7 lam=+0.274 DIVERGED(train)@159 | grad_max=3.5e+17 jac_sv=3.021
[repro] deep s=8 lam=+0.271 DIVERGED(train)@67 | grad_max=2.3e+18 jac_sv=2.968
[repro] deep s=9 lam=+0.282 DIVERGED(train)@225 | grad_max=4.9e+18 jac_sv=2.865
[repro] deep s=10 lam=+0.315 DIVERGED(train)@0 | grad_max=0.0e+00 jac_sv=3.045
[repro] deep s=11 lam=+0.304 DIVERGED(train)@0 | grad_max=0.0e+00 jac_sv=2.949
[repro] edge s=0 lam=+0.015 ok acc=0.7067 | gra

## Part 2 — mitigation sweep on the deep-chaos arm

In [ ]:
CKPT2=os.path.join(RESULTS,"phase22_mitigations.csv")
COLS2=["variant","seed","diverged","first_nan_step","grad_norm_max","acc"]
MIT={
 "baseline":        dict(),
 "smaller_lr_1e-5": dict(lr=1e-5),
 "tight_clip_0.1":  dict(grad_clip=0.1),
 "smoothed_fold":   dict(fold_eps=1e-2),            # removes kink shear, keeps expansion
 "trunc_bptt_8":    dict(tbptt=8),                  # caps Jacobian products -> targets the expansion
 "small_lr+clip":   dict(lr=1e-4,grad_clip=0.25),
}
if os.path.exists(CKPT2):
    rows2=pd.read_csv(CKPT2).to_dict("records"); done2={(r["variant"],int(r["seed"])) for r in rows2}
    print(f"[resume] {len(rows2)} mitigation runs done")
else: rows2=[]; done2=set()
def _f2(): pd.DataFrame(rows2)[COLS2].to_csv(CKPT2,index=False)
plan2=[(v,s) for v in MIT for s in range(CFG.n_seeds_mit)]
for v,s in [k for k in plan2 if k not in done2]:
    r=train_instrumented(CFG,"deep",s,probe=False,**MIT[v])
    rows2.append(dict(variant=v,seed=s,diverged=r["diverged"],first_nan_step=r["first_nan_step"],
                      grad_norm_max=r["grad_norm_max"],acc=r["acc"])); _f2()
    print(f"[mit] {v:16s} s={s} "+("DIVERGED @%d"%r['first_nan_step'] if r['diverged'] else f"ok acc={r['acc']:.4f}"))
RES2=pd.DataFrame(rows2)[COLS2]
print("\nmitigation summary (deep-chaos trained-input):")
g=RES2.groupby("variant")
tab=pd.DataFrame({"diverged":g.diverged.sum().astype(int),"n":g.size(),
                  "div_rate":(g.diverged.mean()).round(2),"mean_acc":g.acc.mean().round(4)})
tab=tab.reindex([v for v in MIT if v in tab.index]); savetable(tab.reset_index(),"phase22_mitigations.csv")
print(tab.to_string())

## Analysis + figure (600 dpi)

In [ ]:
print("MECHANISM SUMMARY")
dd=RES1[RES1.regime=="deep"]; ee=RES1[RES1.regime=="edge"]
print(f"  deep  (lam>0): {int(dd.diverged.sum())}/{len(dd)} diverge "
      f"[init-overflow {int((dd.divergence_phase=='init').sum())}, train-blowup {int((dd.divergence_phase=='train').sum())}] "
      f"| driven-Jacobian top sv {dd.jac_sv_mean.mean():.3f} (>1 = expanding) | grad_norm up to {dd.grad_norm_max.max():.1e}")
print(f"  edge  (lam~0): {int(ee.diverged.sum())}/{len(ee)} diverge | driven-Jacobian top sv {ee.jac_sv_mean.mean():.3f}             | grad_norm up to {ee.grad_norm_max.max():.1e}")
if dd.diverged.any(): print(f"  median step of first non-finite value (deep train-blowups): {dd[dd.divergence_phase=='train'].first_nan_step.median()}")
mt=RES2.groupby("variant").diverged.mean()
print(); print("  mitigation divergence rates:")
for v in MIT:
    if v in mt.index: print(f"    {v:16s} {mt[v]:.2f}")
print(); print("  reading:")
print("  - Forward state is BOUNDED: init-overflow count is ~0 (loss at init is finite, as the")
print("    bounded triangle fold requires). The divergence is NOT unbounded state dynamics.")
print("  - It is the BACKWARD pass: the deep arm's driven Jacobian expands (top sv>1); BPTT multiplies")
print("    ~"+str(CFG.fc_length)+" of these, so the gradient grows geometrically to non-finite. The edge arm stays near-isometric.")
print("  - The smoothed fold is 1-Lipschitz like the hard fold: it removes the kink SHEAR but not the")
print("    EXPANSION, so if it alone does not rescue the arm, the kink is not the dominant cause.")
print("  - Truncated BPTT caps the number of multiplied Jacobians; if THAT rescues it, the cause is")
print("    pinned to the Jacobian product (the positive Lyapunov exponent), exactly as hypothesized.")

fig,ax=plt.subplots(1,2,figsize=(12,4.6))
# (a) grad-norm traces
for _,r in RES1.iterrows():
    tr=json.loads(r["grad_norm_trace"]); c="#B71C1C" if r["regime"]=="deep" else "#2E7D32"
    ax[0].plot(range(len(tr)),tr,color=c,alpha=.5,lw=1)
ax[0].set_yscale("log"); ax[0].set_xlabel("training step"); ax[0].set_ylabel("gradient norm (pre-clip)")
ax[0].set_title("(a) Gradient norm: deep (red) explodes, edge (green) stable",fontsize=11)
from matplotlib.lines import Line2D
ax[0].legend([Line2D([0],[0],color="#B71C1C"),Line2D([0],[0],color="#2E7D32")],["deep","edge"],fontsize=9)
# (b) mitigation divergence rates
vs=[v for v in MIT if v in RES2.variant.unique()]
rates=[RES2[RES2.variant==v].diverged.mean() for v in vs]
ax[1].bar(range(len(vs)),rates,color=(["#B71C1C","#4C72B0","#4C72B0","#E8900C","#2E7D32","#8E24AA"]*2)[:len(vs)])
ax[1].set_xticks(range(len(vs))); ax[1].set_xticklabels(vs,rotation=25,ha="right",fontsize=8)
ax[1].set_ylabel("divergence rate"); ax[1].set_ylim(0,1.05)
ax[1].set_title("(b) Mitigations on the deep-chaos arm",fontsize=11); ax[1].grid(axis="y",alpha=.25)
fig.tight_layout(); savefig(fig,"phase22_divergence.png")

MECHANISM SUMMARY
  deep  (lam>0): 12/12 diverge [init-overflow 0, train-blowup 12] | driven-Jacobian top sv 2.982 (>1 = expanding) | grad_norm up to 1.7e+19
  edge  (lam~0): 0/12 diverge | driven-Jacobian top sv 0.925             | grad_norm up to 7.1e+00
  median step of first non-finite value (deep train-blowups): 118.0

  mitigation divergence rates:
    baseline         1.00
    smaller_lr_1e-5  0.50
    tight_clip_0.1   0.75
    smoothed_fold    0.62
    trunc_bptt_8     0.00
    small_lr+clip    0.38

  reading:
  - Forward state is BOUNDED: init-overflow count is ~0 (loss at init is finite, as the
    bounded triangle fold requires). The divergence is NOT unbounded state dynamics.
  - It is the BACKWARD pass: the deep arm's driven Jacobian expands (top sv>1); BPTT multiplies
    ~64 of these, so the gradient grows geometrically to non-finite. The edge arm stays near-isometric.
  - The smoothed fold is 1-Lipschitz like the hard fold: it removes the kink SHEAR but not the
    EXP

## Rescue if Drive was not mounted

In [ ]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase22_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)